In [2]:
import pandas as pd
import numpy as np
import json
import os
import ast
from pathlib import Path
import torch
from typing import List, Optional
from dataclasses import dataclass

import teradatasql
from sqlalchemy import text, create_engine
from teradataml import create_context, get_context, get_connection, DataFrame, in_schema, copy_to_sql
from teradataml.dataframe.copy_to import copy_to_sql
from dotenv import load_dotenv

import torch
from sentence_transformers import SentenceTransformer

# import sys
# sys.path.append('..')
from constants import (
    CLEANED_TEST_DATA_PATH,
    ENCODED_TEST_DATA_PATH,
    CLEANED_TRAIN_DATA_PATH
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\na255073\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
#from config.settings import (TD_HOST, TD_USER, TD_PASS, TD_DB)
TD_HOST="iteration7-w9og53takluu3v27.env.clearscape.teradata.com"
TD_USER="demo_user"
TD_PASS="n8888888"
TD_DB="DEMO_USER"

In [ ]:
# print(TD_DB,TD_HOST,TD_PASS,TD_USER)
# conn = teradatasql.connect(
#     host=TD_HOST,
#     user=TD_USER,
#     password=TD_PASS,
#     database=TD_DB
# )
# cursor = conn.cursor()
# print("Successfully connected to Teradata!")

In [4]:
conn = teradatasql.connect(
    host=TD_HOST,
    user=TD_USER,
    password=TD_PASS,
    logdata={'CHARSET': 'UTF8'}
)

sqlalchemy_engine = create_engine("teradatasql://", creator=lambda: conn)
create_context(tdsqlengine=sqlalchemy_engine)
print("Connection successful with UTF-8 encoding!")

Connection successful with UTF-8 encoding!


In [10]:
tables_df = DataFrame.from_query(f"""
    SELECT DatabaseName, TableName
    FROM DBC.TablesV
    WHERE DatabaseName = '{TD_DB}'
""")

tables_df

DataBaseName,TableName
demo_user,results_mapped
demo_user,classification_metrics
demo_user,ml___frmqry_v_1755605247458733
demo_user,unique_classes
demo_user,train_embeddings_fc
demo_user,category_embeddings
demo_user,ml___frmqry_v_1755675429469959
demo_user,results
demo_user,ml___frmqry_v_1755608816397687
demo_user,ml__select__1755601586193745


In [11]:
tdf = DataFrame.from_table("original_dataset", schema_name=TD_DB, index_label="Item_Name")
print("Shape of the data:", tdf.shape)

Shape of the data: (4773, 10)


In [12]:
tdf.head(10)

Item_Name,class,Brand,Weight,Number of units,Size of units,Price,T.Price,Pack,Unit
أبو كاس أرز مزة بسمتي هندي 10 كجم,"Rice, Pasta & Pulses",أبو كاس,10كجم,1,None,None,None,كيس,كجم
أجنحة دجاج أطياب - 700جم,Poultry,أطياب,700جم,1,None,None,None,عبوة,جم
أجنحة دجاج حارة أطياب,Poultry,أطياب,None,1,None,None,None,عبوة,None
أحمد تي شاي أخضر نقي - 20 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,None,20,None,None,None,عبوة,None
أحمد تي شاي إيرل جراي - 25 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,None,25,None,None,None,عبوة,None
أحمد تي كولد برو شاي مثلج بالليمون والنعناع 20 كيس,"Tea, Coffee & Hot Drinks",احمد تي,None,20,None,None,None,None,كيس
أحمد تي شاي إيرل جراي - 100 فتلة,"Tea, Coffee & Hot Drinks",أحمد تي,100جم,1,None,None,None,علبة,جم
أجرومونتي صلصة الداترينو 330 جم,"Tins, Jars & Packets",أجرومونتي,330جم,1,None,None,None,عبوة,جم
أبو عوف قهوة تركي محوج وسط 250 جم,"Tea, Coffee & Hot Drinks",أبو عوف,250جم,1,None,None,None,عبوة,جم
أبو علي بابريكا - 85جم,Cooking Ingredients,أبو علي,85جم,1,None,None,None,عبوة,جم


In [13]:
tdf.tdtypes

COLUMN NAME,TYPE
Item_Name,"VARCHAR(length=1024, charset='UNICODE')"
class,"VARCHAR(length=1024, charset='UNICODE')"
Brand,"VARCHAR(length=1024, charset='UNICODE')"
Weight,"VARCHAR(length=1024, charset='UNICODE')"
Number of units,BIGINT()
Size of units,"VARCHAR(length=1024, charset='UNICODE')"
Price,FLOAT()
T.Price,FLOAT()
Pack,"VARCHAR(length=1024, charset='UNICODE')"
Unit,"VARCHAR(length=1024, charset='UNICODE')"


In [14]:
tdf = tdf.dropna(subset=["Item_Name", "class"])

In [15]:
tdf.count()

count_Item_Name,count_class,count_Brand,count_Weight,count_Number of units,count_Size of units,count_Price,count_T.Price,count_Pack,count_Unit
4772,4772,3826,3021,4772,45,278,278,4603,3057


In [ ]:
tdf = tdf.assign(
    Item_Name = tdf.Item_Name.str.lower(),
    **{'class': tdf['class'].str.lower()}
)

In [ ]:
tdf_stripped = tdf.assign(
    Item_Name = tdf.Item_Name.str.strip(),
    **{'class': tdf['class'].str.strip()}
)

In [ ]:
tdf = tdf_stripped.assign(
    Item_Name = tdf_stripped.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", ""),
    **{'class': tdf_stripped['class'].otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")}
)

In [ ]:
cleaned_tdf = tdf[["Item_Name", "class"]]
cleaned_tdf

In [ ]:
cleaned_tdf = cleaned_tdf.sort('row_id')
cleaned_tdf.head(10)

In [ ]:
from teradataml import execute_sql
from teradatasqlalchemy.types import VARCHAR

target_table_name = 'DEMO_USER.cleaned_data'

unicode_type = VARCHAR(length=500, charset='UNICODE')

unicode_tdf = cleaned_tdf.assign(
    Item_Name = cleaned_tdf.Item_Name.cast(unicode_type)
)

source_query = unicode_tdf.show_query()

create_sql = f"""
CREATE TABLE {target_table_name} AS (
    {source_query}
) WITH DATA;
"""

print("--- Generated SQL ---")
print(create_sql)

try:
    print(f"\nAttempting to drop existing table '{target_table_name}'...")
    execute_sql(f"DROP TABLE {target_table_name};")
    print("✔ Previous table dropped.")
except Exception:
    print("Table did not exist, proceeding to create.")

print("\nExecuting CREATE TABLE statement... 🚀")
execute_sql(create_sql)
print(f"✔ Success! Table '{target_table_name}' has been created.")

# Retrieving Cleaned Data from database and creating Embeddings

In [16]:
tdf_embedd = DataFrame.from_table("cleaned_data", schema_name=TD_DB, index_label="row_id")
print("Shape of the data:", tdf_embedd.shape)
tdf_embedd.head(5)

Shape of the data: (4772, 3)


row_id,Item_Name,class
3,أبو كاس أرز مزة بسمتي هندي 10 كجم,rice pasta pulses
5,أجنحة دجاج أطياب 700جم,poultry
4,أجرومونتي صلصة الداترينو 330 جم,tins jars packets
2,أبو عوف قهوة تركي محوج وسط 250 جم,tea coffee hot drinks
1,أبو علي بابريكا 85جم,cooking ingredients


In [17]:
unique_classes_tdf = tdf_embedd.assign(
    drop_columns=True,
    class_name=tdf_embedd['class'].distinct()
)
unique_classes_tdf.count()

count_class_name
32


In [18]:
unique_classes_tdf = tdf_embedd.drop_duplicate(column_names='class')
unique_classes_tdf.count()

count_class
32


In [20]:
# from teradataml import execute_sql
# from teradatasqlalchemy.types import VARCHAR

# target_table_name = 'DEMO_USER.unique_classes'

# # Define the unicode type for the 'class' column.
# unicode_type = VARCHAR(length=500, charset='UNICODE')

# # This part is correct.
# unicode_unique_classes_tdf = unique_classes_tdf.assign(
#     **{'class': unique_classes_tdf['class'].cast(unicode_type)}
# )

# source_query = unicode_unique_classes_tdf.show_query()

# # --- SQL Statement Generation ---

# # 1. DDL with the "class" column quoted and a PRIMARY INDEX defined.
# create_sql = f"""
# CREATE MULTISET TABLE {target_table_name} (
#     class_id INTEGER GENERATED ALWAYS AS IDENTITY (
#         START WITH 1
#         INCREMENT BY 1
#         MINVALUE 1
#         NO MAXVALUE
#         NO CYCLE
#     ),
#     "class" VARCHAR(500) CHARACTER SET UNICODE
# ) PRIMARY INDEX (class_id);
# """

# # 2. DML with the "class" column quoted.
# insert_sql = f"""
# INSERT INTO {target_table_name} ("class")
# {source_query};
# """

# print("--- Generated CREATE TABLE SQL ---")
# print(create_sql)
# print("\n--- Generated INSERT INTO SQL ---")
# print(insert_sql)

# # --- Execution ---

# # Before creating the new table, attempt to drop it if it already exists.
# try:
#     print(f"\nAttempting to drop existing table '{target_table_name}'...")
#     execute_sql(f"DROP TABLE {target_table_name};")
#     print("✔ Previous table dropped.")
# except Exception:
#     print("Table did not exist, proceeding to create.")

# # Execute the CREATE TABLE statement first to build the table structure.
# print("\nExecuting CREATE TABLE statement... 🚀")
# execute_sql(create_sql)
# print(f"✔ Success! Table '{target_table_name}' has been created with an auto-incrementing ID.")

# # Execute the INSERT statement to populate the table with your data.
# print("\nExecuting INSERT statement to populate data... 📊")
# execute_sql(insert_sql)
# print(f"✔ Success! Data has been inserted into '{target_table_name}'.")

In [19]:
classes_tdf = DataFrame.from_table("unique_classes", schema_name=TD_DB)
print("Shape of the data:", classes_tdf.shape)
classes_tdf.head(30)

Shape of the data: (32, 2)


class_id,class
3,sauces dressings condiments
5,dairy eggs
6,cleaning supplies
7,tea coffee hot drinks
10002,biscuits cakes
10003,bakery
10004,poultry
10005,home textile
10006,beef processed meat
10007,nuts dates dried fruits


In [21]:
pandas_df = tdf_embedd.to_pandas()
pandas_df.head(10)

,Item_Name,class
row_id,,
1,أبو علي بابريكا 85جم,cooking ingredients
299,المصرين مكرونه فرن350جم,rice pasta pulses
1785,شويبس جولد خوخ زجاج,soft drinks juices
3172,نسكويك حليب بطعم الفراولة 180 مل,dairy eggs
2,أبو عوف قهوة تركي محوج وسط 250 جم,tea coffee hot drinks
300,المطبخ ارز مصرى 5,rice pasta pulses
1786,شويبس جولد مشروب شعير أناناس، 1 لتر,soft drinks juices
3173,نسكويك مشروب شيكولاتة 11 جم,tea coffee hot drinks
3,أبو كاس أرز مزة بسمتي هندي 10 كجم,rice pasta pulses


In [22]:
# %pip install -U pip
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


# E5 EMBEDDINGS

In [4]:
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Using device: {device}")
print("-" * 30)

Using device: cuda
------------------------------


In [25]:
# --- 3. Embed the "Item_Name" Column (Corrected for Final Format) ---

item_texts = pandas_df["Item_Name"].fillna("").astype(str).tolist()
item_texts_for_model = [f"query: {text}" for text in item_texts]
print(f"Embedding {len(item_texts_for_model)} item names...")

item_embeddings = model.encode(
    item_texts_for_model,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = item_embeddings.shape[1]
emb_cols = [f"v{i}" for i in range(dim)]
item_embeddings_df_vectors = pd.DataFrame(item_embeddings, columns=emb_cols)


identifiers_df = pd.DataFrame({'row_id': pandas_df.index})

# Combine the identifiers with the embeddings
item_embeddings_df = pd.concat([identifiers_df, item_embeddings_df_vectors], axis=1)

print("\n✅ DataFrame with Item_Name Embeddings (item_embeddings_df):")
print(item_embeddings_df.head())

Embedding 4772 item names...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]


✅ DataFrame with Item_Name Embeddings (item_embeddings_df):
   row_id        v0        v1        v2        v3        v4        v5  \
0       1  0.013225  0.014019 -0.004184 -0.046576  0.002445  0.001818   
1     299  0.022322  0.022797 -0.009998 -0.042504  0.004261  0.021432   
2    1785 -0.016994  0.015953 -0.003226 -0.026485  0.019370 -0.001192   
3    3172  0.011858  0.019024  0.019416 -0.044724  0.002927  0.014007   
4       2  0.026973  0.026555  0.003527 -0.057879  0.005757  0.014401   

         v6        v7        v8  ...     v1014     v1015     v1016     v1017  \
0  0.004206  0.056886  0.021367  ... -0.039842 -0.041726  0.023239 -0.021917   
1  0.008610  0.058437  0.036143  ... -0.041176 -0.033954  0.012658 -0.027103   
2 -0.005422  0.061951  0.035064  ... -0.040684 -0.046307 -0.019576 -0.017006   
3 -0.020659  0.061155  0.037989  ... -0.013791 -0.047766  0.026546 -0.031955   
4  0.001295  0.040858  0.028696  ... -0.047304 -0.055964  0.011831 -0.024674   

      v1018     v10

In [55]:
from teradataml import copy_to_sql

if "row_id" not in item_embeddings_df.columns:
    item_embeddings_df.insert(0, "row_id",
        pandas_df["row_id"].astype(int) if "row_id" in pandas_df.columns
        else pandas_df.index.astype(int)
    )

copy_to_sql(
    df=item_embeddings_df,                    
    table_name="e5_item_embeddings",  
    if_exists="replace",
    index=False
)

print("✔ Saved to e5_item_embeddings")


✔ Saved to e5_item_embeddings


In [56]:
tdf_embedd = DataFrame.from_table("e5_item_embeddings", schema_name=TD_DB, index_label="row_id")
print("Shape of the data:", tdf_embedd.shape)
tdf_embedd.head(5)

Shape of the data: (4772, 1025)


row_id,v0,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,v119,v120,v121,v122,v123,v124,v125,v126,v127,v128,v129,v130,v131,v132,v133,v134,v135,v136,v137,v138,v139,v140,v141,v142,v143,v144,v145,v146,v147,v148,v149,v150,v151,v152,v153,v154,v155,v156,v157,v158,v159,v160,v161,v162,v163,v164,v165,v166,v167,v168,v169,v170,v171,v172,v173,v174,v175,v176,v177,v178,v179,v180,v181,v182,v183,v184,v185,v186,v187,v188,v189,v190,v191,v192,v193,v194,v195,v196,v197,v198,v199,v200,v201,v202,v203,v204,v205,v206,v207,v208,v209,v210,v211,v212,v213,v214,v215,v216,v217,v218,v219,v220,v221,v222,v223,v224,v225,v226,v227,v228,v229,v230,v231,v232,v233,v234,v235,v236,v237,v238,v239,v240,v241,v242,v243,v244,v245,v246,v247,v248,v249,v250,v251,v252,v253,v254,v255,v256,v257,v258,v259,v260,v261,v262,v263,v264,v265,v266,v267,v268,v269,v270,v271,v272,v273,v274,v275,v276,v277,v278,v279,v280,v281,v282,v283,v284,v285,v286,v287,v288,v289,v290,v291,v292,v293,v294,v295,v296,v297,v298,v299,v300,v301,v302,v303,v304,v305,v306,v307,v308,v309,v310,v311,v312,v313,v314,v315,v316,v317,v318,v319,v320,v321,v322,v323,v324,v325,v326,v327,v328,v329,v330,v331,v332,v333,v334,v335,v336,v337,v338,v339,v340,v341,v342,v343,v344,v345,v346,v347,v348,v349,v350,v351,v352,v353,v354,v355,v356,v357,v358,v359,v360,v361,v362,v363,v364,v365,v366,v367,v368,v369,v370,v371,v372,v373,v374,v375,v376,v377,v378,v379,v380,v381,v382,v383,v384,v385,v386,v387,v388,v389,v390,v391,v392,v393,v394,v395,v396,v397,v398,v399,v400,v401,v402,v403,v404,v405,v406,v407,v408,v409,v410,v411,v412,v413,v414,v415,v416,v417,v418,v419,v420,v421,v422,v423,v424,v425,v426,v427,v428,v429,v430,v431,v432,v433,v434,v435,v436,v437,v438,v439,v440,v441,v442,v443,v444,v445,v446,v447,v448,v449,v450,v451,v452,v453,v454,v455,v456,v457,v458,v459,v460,v461,v462,v463,v464,v465,v466,v467,v468,v469,v470,v471,v472,v473,v474,v475,v476,v477,v478,v479,v480,v481,v482,v483,v484,v485,v486,v487,v488,v489,v490,v491,v492,v493,v494,v495,v496,v497,v498,v499,v500,v501,v502,v503,v504,v505,v506,v507,v508,v509,v510,v511,v512,v513,v514,v515,v516,v517,v518,v519,v520,v521,v522,v523,v524,v525,v526,v527,v528,v529,v530,v531,v532,v533,v534,v535,v536,v537,v538,v539,v540,v541,v542,v543,v544,v545,v546,v547,v548,v549,v550,v551,v552,v553,v554,v555,v556,v557,v558,v559,v560,v561,v562,v563,v564,v565,v566,v567,v568,v569,v570,v571,v572,v573,v574,v575,v576,v577,v578,v579,v580,v581,v582,v583,v584,v585,v586,v587,v588,v589,v590,v591,v592,v593,v594,v595,v596,v597,v598,v599,v600,v601,v602,v603,v604,v605,v606,v607,v608,v609,v610,v611,v612,v613,v614,v615,v616,v617,v618,v619,v620,v621,v622,v623,v624,v625,v626,v627,v628,v629,v630,v631,v632,v633,v634,v635,v636,v637,v638,v639,v640,v641,v642,v643,v644,v645,v646,v647,v648,v649,v650,v651,v652,v653,v654,v655,v656,v657,v658,v659,v660,v661,v662,v663,v664,v665,v666,v667,v668,v669,v670,v671,v672,v673,v674,v675,v676,v677,v678,v679,v680,v681,v682,v683,v684,v685,v686,v687,v688,v689,v690,v691,v692,v693,v694,v695,v696,v697,v698,v699,v700,v701,v702,v703,v704,v705,v706,v707,v708,v709,v710,v711,v712,v713,v714,v715,v716,v717,v718,v719,v720,v721,v722,v723,v724,v725,v726,v727,v728,v729,v730,v731,v732,v733,v734,v735,v736,v737,v738,v739,v740,v741,v742,v743,v744,v745,v746,v747,v748,v749,v750,v751,v752,v753,v754,v755,v756,v757,v758,v759,v760,v761,v762,v763,v764,v765,v766,v767,v768,v769,v770,v771,v772,v773,v774,v775,v776,v777,v778,v779,v780,v781,v782,v783,v784,v785,v786,v787,v788,v789,v790,v791,v792,v793,v794,v795,v796,v797,v798,v799,v800,v801,v802,v803,v804,v805,v806,v807,v808,v809,v810,v811,v812,v813,v814,v815,v816,v817,v818,v819,v82

In [36]:
pandas_class_df = classes_tdf.to_pandas()
pandas_class_df.head()


,class
class_id,
30006,chocolates sweets desserts
20003,home appliances
10007,nuts dates dried fruits
20001,disposables napkins
5,dairy eggs


In [ ]:
# --- 4. Embed the Unique "class" Column ---
unique_classes = pandas_class_df["class"].dropna().tolist()
class_texts_for_model = [f"passage: {cls}" for cls in unique_classes]
print(f"\nEmbedding {len(class_texts_for_model)} unique classes...")

class_embeddings = model.encode(
    class_texts_for_model,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

class_embeddings_df = pd.DataFrame(class_embeddings, columns=emb_cols)
class_embeddings_df.insert(0, "class", unique_classes)

print("\n✅ DataFrame with Unique Class Embeddings (class_embeddings_df):")
print(class_embeddings_df.head())
print("-" * 30)


Embedding 32 unique classes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DataFrame with Unique Class Embeddings (class_embeddings_df):
                         class        v0        v1        v2        v3  \
0  chocolates sweets  desserts  0.036337  0.040012  0.000632 -0.046787   
1              home appliances  0.042893  0.047407 -0.034455 -0.027766   
2     nuts dates  dried fruits  0.030503  0.029077 -0.033118 -0.043340   
3         disposables  napkins  0.027967  0.030586 -0.023246 -0.019838   
4                  dairy  eggs  0.028216  0.016147 -0.003787 -0.038164   

         v4        v5        v6        v7        v8  ...     v1014     v1015  \
0  0.030495 -0.015984 -0.026864  0.023444  0.032450  ... -0.024458 -0.034311   
1  0.051609 -0.023073 -0.004065  0.036938  0.015718  ... -0.026123 -0.062910   
2  0.023610 -0.008400 -0.009850  0.043228  0.040175  ... -0.034441 -0.039877   
3  0.026475 -0.008925 -0.010903  0.046595  0.026246  ... -0.038213 -0.059463   
4  0.001216  0.009630  0.000624  0.050112  0.028095  ... -0.031846 -0.033261   

      v10

In [47]:
from teradataml import copy_to_sql

copy_to_sql(
    df=class_embeddings_df,                     
    table_name="e5_class_embeddings",   
    if_exists="replace",
    index=False
)

print("✔ Saved to e5_class_embeddings")


✔ Saved to e5_class_embeddings


In [106]:
tdf_embedd = DataFrame.from_table("e5_class_embeddings", schema_name=TD_DB, index_label="class")
print("Shape of the data:", tdf_embedd.shape)
tdf_embedd.head(5)

Shape of the data: (32, 1025)


class,v0,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,v119,v120,v121,v122,v123,v124,v125,v126,v127,v128,v129,v130,v131,v132,v133,v134,v135,v136,v137,v138,v139,v140,v141,v142,v143,v144,v145,v146,v147,v148,v149,v150,v151,v152,v153,v154,v155,v156,v157,v158,v159,v160,v161,v162,v163,v164,v165,v166,v167,v168,v169,v170,v171,v172,v173,v174,v175,v176,v177,v178,v179,v180,v181,v182,v183,v184,v185,v186,v187,v188,v189,v190,v191,v192,v193,v194,v195,v196,v197,v198,v199,v200,v201,v202,v203,v204,v205,v206,v207,v208,v209,v210,v211,v212,v213,v214,v215,v216,v217,v218,v219,v220,v221,v222,v223,v224,v225,v226,v227,v228,v229,v230,v231,v232,v233,v234,v235,v236,v237,v238,v239,v240,v241,v242,v243,v244,v245,v246,v247,v248,v249,v250,v251,v252,v253,v254,v255,v256,v257,v258,v259,v260,v261,v262,v263,v264,v265,v266,v267,v268,v269,v270,v271,v272,v273,v274,v275,v276,v277,v278,v279,v280,v281,v282,v283,v284,v285,v286,v287,v288,v289,v290,v291,v292,v293,v294,v295,v296,v297,v298,v299,v300,v301,v302,v303,v304,v305,v306,v307,v308,v309,v310,v311,v312,v313,v314,v315,v316,v317,v318,v319,v320,v321,v322,v323,v324,v325,v326,v327,v328,v329,v330,v331,v332,v333,v334,v335,v336,v337,v338,v339,v340,v341,v342,v343,v344,v345,v346,v347,v348,v349,v350,v351,v352,v353,v354,v355,v356,v357,v358,v359,v360,v361,v362,v363,v364,v365,v366,v367,v368,v369,v370,v371,v372,v373,v374,v375,v376,v377,v378,v379,v380,v381,v382,v383,v384,v385,v386,v387,v388,v389,v390,v391,v392,v393,v394,v395,v396,v397,v398,v399,v400,v401,v402,v403,v404,v405,v406,v407,v408,v409,v410,v411,v412,v413,v414,v415,v416,v417,v418,v419,v420,v421,v422,v423,v424,v425,v426,v427,v428,v429,v430,v431,v432,v433,v434,v435,v436,v437,v438,v439,v440,v441,v442,v443,v444,v445,v446,v447,v448,v449,v450,v451,v452,v453,v454,v455,v456,v457,v458,v459,v460,v461,v462,v463,v464,v465,v466,v467,v468,v469,v470,v471,v472,v473,v474,v475,v476,v477,v478,v479,v480,v481,v482,v483,v484,v485,v486,v487,v488,v489,v490,v491,v492,v493,v494,v495,v496,v497,v498,v499,v500,v501,v502,v503,v504,v505,v506,v507,v508,v509,v510,v511,v512,v513,v514,v515,v516,v517,v518,v519,v520,v521,v522,v523,v524,v525,v526,v527,v528,v529,v530,v531,v532,v533,v534,v535,v536,v537,v538,v539,v540,v541,v542,v543,v544,v545,v546,v547,v548,v549,v550,v551,v552,v553,v554,v555,v556,v557,v558,v559,v560,v561,v562,v563,v564,v565,v566,v567,v568,v569,v570,v571,v572,v573,v574,v575,v576,v577,v578,v579,v580,v581,v582,v583,v584,v585,v586,v587,v588,v589,v590,v591,v592,v593,v594,v595,v596,v597,v598,v599,v600,v601,v602,v603,v604,v605,v606,v607,v608,v609,v610,v611,v612,v613,v614,v615,v616,v617,v618,v619,v620,v621,v622,v623,v624,v625,v626,v627,v628,v629,v630,v631,v632,v633,v634,v635,v636,v637,v638,v639,v640,v641,v642,v643,v644,v645,v646,v647,v648,v649,v650,v651,v652,v653,v654,v655,v656,v657,v658,v659,v660,v661,v662,v663,v664,v665,v666,v667,v668,v669,v670,v671,v672,v673,v674,v675,v676,v677,v678,v679,v680,v681,v682,v683,v684,v685,v686,v687,v688,v689,v690,v691,v692,v693,v694,v695,v696,v697,v698,v699,v700,v701,v702,v703,v704,v705,v706,v707,v708,v709,v710,v711,v712,v713,v714,v715,v716,v717,v718,v719,v720,v721,v722,v723,v724,v725,v726,v727,v728,v729,v730,v731,v732,v733,v734,v735,v736,v737,v738,v739,v740,v741,v742,v743,v744,v745,v746,v747,v748,v749,v750,v751,v752,v753,v754,v755,v756,v757,v758,v759,v760,v761,v762,v763,v764,v765,v766,v767,v768,v769,v770,v771,v772,v773,v774,v775,v776,v777,v778,v779,v780,v781,v782,v783,v784,v785,v786,v787,v788,v789,v790,v791,v792,v793,v794,v795,v796,v797,v798,v799,v800,v801,v802,v803,v804,v805,v806,v807,v808,v809,v810,v811,v812,v813,v814,v815,v816,v817,v818,v819,v820

In [96]:
tdf_embedd = DataFrame.from_table("original_dataset", schema_name=TD_DB)
print("Shape of the data:", tdf_embedd.shape)
tdf_embedd.head(5)

Shape of the data: (4773, 10)


Item_Name,class,Brand,Weight,Number of units,Size of units,Price,T.Price,Pack,Unit
أبو كاس أرز مزة بسمتي هندي 10 كجم,"Rice, Pasta & Pulses",أبو كاس,10كجم,1,None,None,None,كيس,كجم
أجنحة دجاج أطياب - 700جم,Poultry,أطياب,700جم,1,None,None,None,عبوة,جم
أجرومونتي صلصة الداترينو 330 جم,"Tins, Jars & Packets",أجرومونتي,330جم,1,None,None,None,عبوة,جم
أبو عوف قهوة تركي محوج وسط 250 جم,"Tea, Coffee & Hot Drinks",أبو عوف,250جم,1,None,None,None,عبوة,جم
أبو علي بابريكا - 85جم,Cooking Ingredients,أبو علي,85جم,1,None,None,None,عبوة,جم


In [97]:
raw_classes_tdf = tdf_embedd.assign(
    drop_columns=True,
    class_name=tdf_embedd['class'].distinct()
)
raw_classes_tdf.count()

count_class_name
32


In [99]:
pandas_raw_class_df = raw_classes_tdf.to_pandas()
pandas_raw_class_df.head(10)

,class_name
0,Baby Care
1,Furniture
2,Cooking Ingredients
3,None
4,Bakery
5,Soft Drinks & Juices
6,Disposables & Napkins
7,Beef & Processed Meat
8,Beef & Lamb Meat
9,Sweets & Desserts


In [102]:
# --- 4. Embed the Unique "class" Column ---
raw_unique_classes = pandas_raw_class_df["class_name"].dropna().tolist()
class_texts_for_model = [f"passage: {cls}" for cls in raw_unique_classes]
print(f"\nEmbedding {len(class_texts_for_model)} unique classes...")

class_embeddings = model.encode(
    class_texts_for_model,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

class_embeddings_df = pd.DataFrame(class_embeddings, columns=emb_cols)
class_embeddings_df.insert(0, "class", raw_unique_classes)

print("\n✅ DataFrame with  raw Unique Class Embeddings (class_embeddings_df):")
print(class_embeddings_df.head())
print("-" * 30)


Embedding 32 unique classes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DataFrame with  raw Unique Class Embeddings (class_embeddings_df):
                  class        v0        v1        v2        v3        v4  \
0             Baby Care  0.031241  0.026698 -0.017527 -0.009856  0.006616   
1             Furniture  0.043846  0.034735 -0.042679 -0.006470  0.022016   
2   Cooking Ingredients  0.054189  0.049481 -0.000571 -0.026300  0.028475   
3                Bakery  0.033040  0.040331  0.001702 -0.047357  0.014604   
4  Soft Drinks & Juices  0.032933  0.039830 -0.007727 -0.028317  0.019316   

         v5        v6        v7        v8  ...     v1014     v1015     v1016  \
0 -0.006016 -0.015202  0.047995  0.043667  ... -0.015439 -0.045744  0.029302   
1 -0.028083 -0.012778  0.038191  0.019790  ...  0.001735 -0.052765  0.017229   
2 -0.048325  0.006177  0.045085  0.007401  ... -0.031929 -0.056914  0.029559   
3  0.008822 -0.032445  0.023149  0.022589  ... -0.024300 -0.051492  0.007244   
4 -0.019084 -0.017476  0.033383  0.041971  ... -0.024674 -0.058454 

In [103]:
from teradataml import copy_to_sql

copy_to_sql(
    df=class_embeddings_df,                     
    table_name="e5_raw_class_embeddings",   
    if_exists="replace",
    index=False
)

print("✔ Saved to e5_raw_class_embeddings")


✔ Saved to e5_raw_class_embeddings


In [105]:
tdf_embedd = DataFrame.from_table("e5_raw_class_embeddings", schema_name=TD_DB, index_label="class")
print("Shape of the data:", tdf_embedd.shape)
tdf_embedd.head(5)

Shape of the data: (32, 1025)


class,v0,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11,v12,v13,v14,v15,v16,v17,v18,v19,v20,v21,v22,v23,v24,v25,v26,v27,v28,v29,v30,v31,v32,v33,v34,v35,v36,v37,v38,v39,v40,v41,v42,v43,v44,v45,v46,v47,v48,v49,v50,v51,v52,v53,v54,v55,v56,v57,v58,v59,v60,v61,v62,v63,v64,v65,v66,v67,v68,v69,v70,v71,v72,v73,v74,v75,v76,v77,v78,v79,v80,v81,v82,v83,v84,v85,v86,v87,v88,v89,v90,v91,v92,v93,v94,v95,v96,v97,v98,v99,v100,v101,v102,v103,v104,v105,v106,v107,v108,v109,v110,v111,v112,v113,v114,v115,v116,v117,v118,v119,v120,v121,v122,v123,v124,v125,v126,v127,v128,v129,v130,v131,v132,v133,v134,v135,v136,v137,v138,v139,v140,v141,v142,v143,v144,v145,v146,v147,v148,v149,v150,v151,v152,v153,v154,v155,v156,v157,v158,v159,v160,v161,v162,v163,v164,v165,v166,v167,v168,v169,v170,v171,v172,v173,v174,v175,v176,v177,v178,v179,v180,v181,v182,v183,v184,v185,v186,v187,v188,v189,v190,v191,v192,v193,v194,v195,v196,v197,v198,v199,v200,v201,v202,v203,v204,v205,v206,v207,v208,v209,v210,v211,v212,v213,v214,v215,v216,v217,v218,v219,v220,v221,v222,v223,v224,v225,v226,v227,v228,v229,v230,v231,v232,v233,v234,v235,v236,v237,v238,v239,v240,v241,v242,v243,v244,v245,v246,v247,v248,v249,v250,v251,v252,v253,v254,v255,v256,v257,v258,v259,v260,v261,v262,v263,v264,v265,v266,v267,v268,v269,v270,v271,v272,v273,v274,v275,v276,v277,v278,v279,v280,v281,v282,v283,v284,v285,v286,v287,v288,v289,v290,v291,v292,v293,v294,v295,v296,v297,v298,v299,v300,v301,v302,v303,v304,v305,v306,v307,v308,v309,v310,v311,v312,v313,v314,v315,v316,v317,v318,v319,v320,v321,v322,v323,v324,v325,v326,v327,v328,v329,v330,v331,v332,v333,v334,v335,v336,v337,v338,v339,v340,v341,v342,v343,v344,v345,v346,v347,v348,v349,v350,v351,v352,v353,v354,v355,v356,v357,v358,v359,v360,v361,v362,v363,v364,v365,v366,v367,v368,v369,v370,v371,v372,v373,v374,v375,v376,v377,v378,v379,v380,v381,v382,v383,v384,v385,v386,v387,v388,v389,v390,v391,v392,v393,v394,v395,v396,v397,v398,v399,v400,v401,v402,v403,v404,v405,v406,v407,v408,v409,v410,v411,v412,v413,v414,v415,v416,v417,v418,v419,v420,v421,v422,v423,v424,v425,v426,v427,v428,v429,v430,v431,v432,v433,v434,v435,v436,v437,v438,v439,v440,v441,v442,v443,v444,v445,v446,v447,v448,v449,v450,v451,v452,v453,v454,v455,v456,v457,v458,v459,v460,v461,v462,v463,v464,v465,v466,v467,v468,v469,v470,v471,v472,v473,v474,v475,v476,v477,v478,v479,v480,v481,v482,v483,v484,v485,v486,v487,v488,v489,v490,v491,v492,v493,v494,v495,v496,v497,v498,v499,v500,v501,v502,v503,v504,v505,v506,v507,v508,v509,v510,v511,v512,v513,v514,v515,v516,v517,v518,v519,v520,v521,v522,v523,v524,v525,v526,v527,v528,v529,v530,v531,v532,v533,v534,v535,v536,v537,v538,v539,v540,v541,v542,v543,v544,v545,v546,v547,v548,v549,v550,v551,v552,v553,v554,v555,v556,v557,v558,v559,v560,v561,v562,v563,v564,v565,v566,v567,v568,v569,v570,v571,v572,v573,v574,v575,v576,v577,v578,v579,v580,v581,v582,v583,v584,v585,v586,v587,v588,v589,v590,v591,v592,v593,v594,v595,v596,v597,v598,v599,v600,v601,v602,v603,v604,v605,v606,v607,v608,v609,v610,v611,v612,v613,v614,v615,v616,v617,v618,v619,v620,v621,v622,v623,v624,v625,v626,v627,v628,v629,v630,v631,v632,v633,v634,v635,v636,v637,v638,v639,v640,v641,v642,v643,v644,v645,v646,v647,v648,v649,v650,v651,v652,v653,v654,v655,v656,v657,v658,v659,v660,v661,v662,v663,v664,v665,v666,v667,v668,v669,v670,v671,v672,v673,v674,v675,v676,v677,v678,v679,v680,v681,v682,v683,v684,v685,v686,v687,v688,v689,v690,v691,v692,v693,v694,v695,v696,v697,v698,v699,v700,v701,v702,v703,v704,v705,v706,v707,v708,v709,v710,v711,v712,v713,v714,v715,v716,v717,v718,v719,v720,v721,v722,v723,v724,v725,v726,v727,v728,v729,v730,v731,v732,v733,v734,v735,v736,v737,v738,v739,v740,v741,v742,v743,v744,v745,v746,v747,v748,v749,v750,v751,v752,v753,v754,v755,v756,v757,v758,v759,v760,v761,v762,v763,v764,v765,v766,v767,v768,v769,v770,v771,v772,v773,v774,v775,v776,v777,v778,v779,v780,v781,v782,v783,v784,v785,v786,v787,v788,v789,v790,v791,v792,v793,v794,v795,v796,v797,v798,v799,v800,v801,v802,v803,v804,v805,v806,v807,v808,v809,v810,v811,v812,v813,v814,v815,v816,v817,v818,v819,v820

In [94]:
# # --- setup & context ---
# from teradataml import create_context, remove_context, get_context, DataFrame, execute_sql
# from teradatasqlalchemy.types import VARCHAR

# # fill in your creds (or import from your settings)
# # TD_HOST, TD_USER, TD_PASS = ...

# # Always create a teradataml context before execute_sql / DataFrame
# create_context(host=TD_HOST, username=TD_USER, password=TD_PASS)  # add logmech=... if your site requires

# target_table_name = 'DEMO_USER.raw_unique_classes'

# # --- build source from ORIGINAL dataset ---
# # point to the table that has the UN-CLEANED/raw class names
# orig_tdf =  DataFrame.from_table("original_dataset", schema_name=TD_DB)       # <-- change if your table name is different
# unique_classes_tdf = orig_tdf.select("class")

# # ensure UNICODE type
# unicode_type = VARCHAR(length=500, charset='UNICODE')
# raw_classes_tdf = unique_classes_tdf.assign(**{
#     "class": unique_classes_tdf["class"].cast(unicode_type)
# })

# # this query now comes from ORIGINAL dataset (not cleaned_data)
# source_query = raw_classes_tdf.show_query()

# # --- SQL statements ---
# create_sql = f"""
# CREATE MULTISET TABLE {target_table_name} (
#     class_id INTEGER GENERATED ALWAYS AS IDENTITY (
#         START WITH 1
#         INCREMENT BY 1
#         MINVALUE 1
#         NO MAXVALUE
#         NO CYCLE
#     ),
#     "class" VARCHAR(500) CHARACTER SET UNICODE
# ) PRIMARY INDEX (class_id);
# """

# insert_sql = f"""
# INSERT INTO {target_table_name} ("class")
# {source_query};
# """

# print("--- Generated CREATE TABLE SQL ---\n", create_sql)
# print("\n--- Generated INSERT INTO SQL ---\n", insert_sql)

# # --- execute ---
# try:
#     print(f"\nAttempting to drop existing table '{target_table_name}'...")
#     execute_sql(f"DROP TABLE {target_table_name};")
#     print("✔ Previous table dropped.")
# except Exception:
#     print("Table did not exist, proceeding to create.")

# print("\nExecuting CREATE TABLE statement... 🚀")
# execute_sql(create_sql)
# print(f"✔ Created {target_table_name}")

# print("\nExecuting INSERT statement to populate data... 📊")
# execute_sql(insert_sql)
# print(f"✔ Inserted distinct raw classes into {target_table_name}")

# # (optional) verify
# check = DataFrame(target_table_name).sample(n=5)
# print("\nSample rows from raw_unique_classes:")
# print(check.to_pandas())

# QWEN EMBEDDINGS

In [5]:
MODEL_NAME ="Qwen/Qwen3-Embedding-8B"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Using device: {device}")
print("-" * 30)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 24.41 GiB is allocated by PyTorch, and 1.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Cosine Similarity

In [107]:
import re
import teradatasql
import pandas as pd
import torch

# Expect TD_HOST, TD_USER, TD_PASS, TD_DB to be defined in your env/settings
ITEM_EMB = f"{TD_DB}.e5_item_embeddings"      # row_id, v0..v1023
REF_EMB  = f"{TD_DB}.e5_raw_class_embeddings"     # class/gpc_id/..., v0..v1023
RESULT   = f"{TD_DB}.e5_raw_cosine_similarity"

device = "cuda" if torch.cuda.is_available() else "cpu"

def quote_ident(col: str) -> str:
    # Teradata standard identifier quoting
    return '"' + col.replace('"', '""') + '"'

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    # --- discover columns in REF_EMB ---
    ref_cols = pd.read_sql(f"""
        SELECT ColumnName
        FROM DBC.ColumnsV
        WHERE UPPER(DatabaseName)=UPPER('{TD_DB}')
          AND UPPER(TableName)=UPPER('{REF_EMB.split('.')[-1]}')
        ORDER BY ColumnId
    """, con)["ColumnName"].tolist()

    # --- pick an ID column robustly (case-insensitive) ---
    cands = {"class", "gpc_id", "label_id", "orig_label_id", "id", "row_id"}
    try:
        ref_id_col = next(c for c in ref_cols if c.lower() in cands)
    except StopIteration:
        raise ValueError(f"No suitable ID column in {REF_EMB}. Found: {ref_cols}")

    # --- discover columns in ITEM_EMB ---
    item_cols = pd.read_sql(f"""
        SELECT ColumnName
        FROM DBC.ColumnsV
        WHERE UPPER(DatabaseName)=UPPER('{TD_DB}')
          AND UPPER(TableName)=UPPER('{ITEM_EMB.split('.')[-1]}')
        ORDER BY ColumnId
    """, con)["ColumnName"].tolist()

    if "row_id" not in item_cols:
        raise ValueError(f"{ITEM_EMB} must contain 'row_id'. Found: {item_cols[:10]}...")

    # --- shared v* feature columns (keep consistent order) ---
    item_feats = [c for c in item_cols if c.startswith("v")]
    ref_feats  = [c for c in ref_cols  if c.startswith("v")]
    feats = sorted(
        set(item_feats).intersection(ref_feats),
        key=lambda x: int(x[1:]) if x[1:].isdigit() else x
    )
    if not feats:
        raise ValueError("No shared v* feature columns between items and refs.")
    vlist = ", ".join(feats)

    # --- read data; quote the ref id if needed (e.g., "class") ---
    q_ref_id = quote_ident(ref_id_col)  # handles reserved words like class
    items = pd.read_sql(f"SELECT row_id, {vlist} FROM {ITEM_EMB}", con)
    refs  = pd.read_sql(f"SELECT {q_ref_id} AS ref_id, {vlist} FROM {REF_EMB}", con)

# --- torch tensors on the chosen device ---
# Ensure integer dtype for ids; some Teradata drivers return object dtype
I_ids = torch.tensor(pd.to_numeric(items["row_id"], errors="raise").values, dtype=torch.long, device=device)
R_ids = refs["ref_id"].astype(str).tolist()

I = torch.tensor(items[feats].values, dtype=torch.float32, device=device)
R = torch.tensor(refs[feats].values,  dtype=torch.float32, device=device)

# --- cosine similarity via L2-normalization ---
I = torch.nn.functional.normalize(I, p=2, dim=1)
R = torch.nn.functional.normalize(R, p=2, dim=1)

scores = I @ R.T  # [n_items, n_refs]
best_score, best_idx = scores.max(dim=1)

pred_df = pd.DataFrame({
    "row_id": I_ids.cpu().numpy(),
    "orig_label_id": [R_ids[i] for i in best_idx.cpu().numpy()],
    "score": best_score.detach().cpu().numpy().round(6),
})

# --- write predictions back to Teradata ---
with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    cur = con.cursor()
    try:
        cur.execute(f"DROP TABLE {RESULT}")
    except Exception:
        pass
    cur.execute(f"""
        CREATE MULTISET TABLE {RESULT} (
            row_id BIGINT,
            orig_label_id VARCHAR(1024) CHARACTER SET UNICODE,
            score FLOAT
        )
        PRIMARY INDEX (row_id)
    """)
    # executemany expects iterable of tuples
    cur.executemany(
        f"INSERT INTO {RESULT} (row_id, orig_label_id, score) VALUES (?,?,?)",
        list(map(tuple, pred_df.values))
    )

print("Done. Top‑1 cosine predictions computed client‑side and saved to", RESULT)


C:\Users\na255073\AppData\Local\Temp\ipykernel_35340\327917800.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ref_cols = pd.read_sql(f"""
C:\Users\na255073\AppData\Local\Temp\ipykernel_35340\327917800.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  item_cols = pd.read_sql(f"""
C:\Users\na255073\AppData\Local\Temp\ipykernel_35340\327917800.py:59: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  items = pd.read_sql(f"SELECT row_id, {vlist} FROM {ITEM_EMB}", con)
C:\Users\na255073\AppData\Local\Temp\ipykernel_35340\327

Done. Top‑1 cosine predictions computed client‑side and saved to DEMO_USER.e5_raw_cosine_similarity


# EVALUATION

f1 before cleaning the class column

In [7]:
import pandas as pd
from sklearn.metrics import f1_score, classification_report
import teradatasql

GT_TBL   = f"{TD_DB}.original_dataset"     # ground truth with the raw/non-cleaned class
PRED_TBL =  f"{TD_DB}.e5_raw_cosine_similarity"                      # your predictions table: row_id, orig_label_id, score
ITEM_EMB = f"{TD_DB}.e5_item_embeddings"   # used only as a row_id -> external_id map if it has that column

ID_CANDIDATES = ["row_id","id","item_id","product_id","sku_id","sku","record_id","source_id"]

def get_cols(con, fqn):
    db, tbl = fqn.split(".")
    q = f"""
      SELECT ColumnName
      FROM DBC.ColumnsV
      WHERE UPPER(DatabaseName)=UPPER('{db}') AND UPPER(TableName)=UPPER('{tbl}')
      ORDER BY ColumnId
    """
    return pd.read_sql(q, con)["ColumnName"].tolist()

def quote_ident(col):
    return '"' + col.replace('"','""') + '"'

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    # --- discover columns ---
    gt_cols   = get_cols(con, GT_TBL)
    emb_cols  = get_cols(con, ITEM_EMB)
    pred_cols = get_cols(con, PRED_TBL)

    # label column in ground truth
    try:
        class_col = next(c for c in gt_cols if c.lower() == "class")
    except StopIteration:
        raise ValueError(f"No 'class' column (any casing) found in {GT_TBL}. Found: {gt_cols}")

    # pick an ID column in ground truth (prefer non-row_id; fall back to row_id if present)
    gt_id_col = None
    for cand in ID_CANDIDATES:
        if cand in gt_cols:
            gt_id_col = cand
            break
        # case-insensitive match
        lower_map = {c.lower(): c for c in gt_cols}
        if cand in lower_map:
            gt_id_col = lower_map[cand]
            break
    if gt_id_col is None:
        raise ValueError(
            f"Could not find any id column in {GT_TBL}. "
            f"Tried: {ID_CANDIDATES}. Found: {gt_cols}"
        )

    # read truth (id + class)
    q_truth = f'SELECT {quote_ident(gt_id_col)} AS gt_id, {quote_ident(class_col)} AS gt_class FROM {GT_TBL}'
    truth   = pd.read_sql(q_truth, con)

    # predictions must have row_id
    if "row_id" not in pred_cols:
        raise ValueError(f"{PRED_TBL} must contain 'row_id'. Found: {pred_cols}")
    preds = pd.read_sql(f"SELECT row_id, orig_label_id FROM {PRED_TBL}", con)

    # If ground truth id is already 'row_id', we can join directly
    if gt_id_col.lower() == "row_id":
        df = truth.merge(preds, left_on="gt_id", right_on="row_id", how="inner")
    else:
        # We need a mapping from row_id -> the ground truth id (gt_id_col).
        # Try to build it from ITEM_EMB if that ID exists there.
        emb_has_gt_id = any(c.lower() == gt_id_col.lower() for c in emb_cols)
        if not emb_has_gt_id:
            raise ValueError(
                f"No shared ID to join predictions to ground truth.\n"
                f"- Ground truth id column: {gt_id_col} (in {GT_TBL})\n"
                f"- {ITEM_EMB} columns: {emb_cols}\n\n"
                f"Fix: Add a mapping table or add '{gt_id_col}' to {ITEM_EMB} so we can map row_id → {gt_id_col}."
            )

        # read mapping from embeddings table
        # find exact-cased column name in emb_cols
        emb_id_exact = next(c for c in emb_cols if c.lower() == gt_id_col.lower())
        map_df = pd.read_sql(
            f'SELECT row_id, {quote_ident(emb_id_exact)} AS gt_id FROM {ITEM_EMB}',
            con
        )

        # join preds -> map (row_id -> gt_id) -> truth (gt_id -> gt_class)
        preds_mapped = preds.merge(map_df, on="row_id", how="inner")
        df = preds_mapped.merge(truth, on="gt_id", how="inner")

# sanity checks
if df.empty:
    raise ValueError("After joining, no rows remain. Check your ID mapping/join keys.")

y_true = df["gt_class"].astype(str)
y_pred = df["orig_label_id"].astype(str)

print("F1 (macro):   ", f1_score(y_true, y_pred, average="macro"))
print("F1 (weighted):", f1_score(y_true, y_pred, average="weighted"))
print("\nPer-class report:\n")
print(classification_report(y_true, y_pred, digits=4))


C:\Users\na255073\AppData\Local\Temp\ipykernel_12344\3696659596.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(q, con)["ColumnName"].tolist()
C:\Users\na255073\AppData\Local\Temp\ipykernel_12344\3696659596.py:55: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  truth   = pd.read_sql(q_truth, con)
C:\Users\na255073\AppData\Local\Temp\ipykernel_12344\3696659596.py:60: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  preds = pd.read_sql(f"SELECT row_id, orig_label_id FROM {PRED_TBL}", con)


F1 (macro):    0.018725056851383058
F1 (weighted): 0.04302881217823612

Per-class report:

                                   precision    recall  f1-score   support

                        Baby Care     0.0000    0.0000    0.0000         2
                           Bakery     0.0526    0.0041    0.0077       241
                 Beef & Lamb Meat     0.0000    0.0000    0.0000         1
            Beef & Processed Meat     0.0280    0.0150    0.0195       267
                 Biscuits & Cakes     0.0914    0.0511    0.0655       333
         Candles & Air Fresheners     0.0000    0.0000    0.0000         1
                 Chips & Crackers     0.0720    0.0812    0.0763       308
                 Chips & crackers     0.0000    0.0000    0.0000         1
    Chocolates, Sweets & Desserts     0.0690    0.0613    0.0649       261
                Cleaning Supplies     0.0769    0.0294    0.0426       170
Condiments, Dressings & Marinades     0.0000    0.0000    0.0000         9
        

c:\Users\na255073\Documents\Product-GPC-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\na255073\Documents\Product-GPC-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\na255073\Documents\Product-GPC-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

f1 after cleaning the class column

In [9]:
import pandas as pd
from sklearn.metrics import f1_score, classification_report
import teradatasql

GT_TBL = f"{TD_DB}.cleaned_data"        # <- ground-truth table with row_id + class
PRED_TBL = f"{TD_DB}.e5_cosine_similarity"                       # <- predictions table we just built

def get_cols(con, fqn):
    db, tbl = fqn.split(".")
    q = f"""
      SELECT ColumnName
      FROM DBC.ColumnsV
      WHERE UPPER(DatabaseName)=UPPER('{db}') AND UPPER(TableName)=UPPER('{tbl}')
      ORDER BY ColumnId
    """
    return pd.read_sql(q, con)["ColumnName"].tolist()

def quote_ident(col):
    # Double-quote for Teradata identifiers; preserve case
    return '"' + col.replace('"', '""') + '"'

with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
    # discover exact casing of the ground-truth label column named class
    gt_cols = get_cols(con, GT_TBL)
    class_col = next(c for c in gt_cols if c.lower() == "class")  # raises if not found
    q_truth = f'SELECT row_id, {quote_ident(class_col)} AS gt_class FROM {GT_TBL}'
    truth = pd.read_sql(q_truth, con)

    preds = pd.read_sql(f"SELECT row_id, orig_label_id FROM {PRED_TBL}", con)

# join and score
df = truth.merge(preds, on="row_id", how="inner")
y_true = df["gt_class"].astype(str)
y_pred = df["orig_label_id"].astype(str)

print("F1 (macro):   ", f1_score(y_true, y_pred, average="macro"))
print("F1 (weighted):", f1_score(y_true, y_pred, average="weighted"))
print("\nPer-class report:\n")
print(classification_report(y_true, y_pred, digits=4))

C:\Users\na255073\AppData\Local\Temp\ipykernel_12344\3087542280.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(q, con)["ColumnName"].tolist()
C:\Users\na255073\AppData\Local\Temp\ipykernel_12344\3087542280.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  truth = pd.read_sql(q_truth, con)
C:\Users\na255073\AppData\Local\Temp\ipykernel_12344\3087542280.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  preds = pd.read_sql(f"SELECT row_id, orig_label_id FROM {PRED_TBL}", con)


F1 (macro):    0.19750934185606844
F1 (weighted): 0.3634338697730764

Per-class report:

                                 precision    recall  f1-score   support

                      baby care     0.0000    0.0000    0.0000         2
                         bakery     0.7143    0.0207    0.0402       242
                beef  lamb meat     0.0000    0.0000    0.0000         1
           beef  processed meat     0.5517    0.4794    0.5130       267
                biscuits  cakes     0.5183    0.4685    0.4921       333
        candles  air fresheners     0.0000    0.0000    0.0000         1
                chips  crackers     0.3182    0.6796    0.4334       309
    chocolates sweets  desserts     0.4565    0.4828    0.4693       261
              cleaning supplies     1.0000    0.2765    0.4332       170
condiments dressings  marinades     0.0089    0.1111    0.0165         9
            cooking ingredients     0.2500    0.0036    0.0071       278
                    dairy  eggs   

c:\Users\na255073\Documents\Product-GPC-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\na255073\Documents\Product-GPC-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\na255073\Documents\Product-GPC-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [79]:
import pandas as pd
# Support per class (true distribution)
support = y_true.value_counts().rename("support_true")
# What you actually predicted
pred_counts = y_pred.value_counts().rename("pred_count")

diag = pd.concat([support, pred_counts], axis=1).fillna(0).astype(int)
diag["never_predicted"] = (diag["pred_count"] == 0)
diag = diag.sort_values("support_true", ascending=False)

print("Classes never predicted (top 20 by true support):")
print(diag[diag.never_predicted].head(20))

print("\nLargest mismatch (top 20 by abs(support - pred)):")
mismatch = (diag["support_true"] - diag["pred_count"]).abs().sort_values(ascending=False)
print(diag.loc[mismatch.index].head(20))


Classes never predicted (top 20 by true support):
           support_true  pred_count  never_predicted
furniture            89           0             True

Largest mismatch (top 20 by abs(support - pred)):
                                 support_true  pred_count  never_predicted
rice pasta  pulses                        249        1129            False
chips  crackers                           309         660            False
soft drinks  juices                       563         256            False
cooking ingredients                       278           4            False
bakery                                    242           7            False
vegetables  fruits                        259          36            False
poultry                                   258          43            False
sauces dressings  condiments              326         131            False
dairy  eggs                               340         147            False
nuts dates  dried fruits                   

In [68]:
# import teradatasql
# import pandas as pd

# # --- CONFIG: update only if your table names differ ---
# ITEM_EMB_SRC  = f"{TD_DB}.e5_item_embeddings"      # has: row_id, v0..v1023
# CLASS_EMB_SRC = f"{TD_DB}.e5_class_embeddings	"     # has: "class", v0..v1023   (or your class-emb table)
# CLASS_MAP     = f"{TD_DB}.unique_classes"          # has: class_id, class      (human class names)
# # staging/outputs
# REF_EMB       = f"{TD_DB}.e5_class_embeddings_with_id"     # we (re)create: class_id, v0..v1023
# RESULT        = f"{TD_DB}.results_mapped"          # final: row_id, class_id, score

# def get_columns(con, db, table):
#     sql = f"""
#     SELECT ColumnName
#     FROM DBC.ColumnsV
#     WHERE DatabaseName = '{db}'
#       AND TableName    = '{table.split('.')[-1]}'
#     ORDER BY ColumnId
#     """
#     return pd.read_sql(sql, con)["ColumnName"].tolist()

# def sorted_vcols(cols):
#     vcols = [c for c in cols if c.startswith('v')]
#     # numeric sort even if some odd names sneak in
#     return sorted(vcols, key=lambda x: int(x[1:]) if x[1:].isdigit() else x)

# print("Opening fresh connection…")
# with teradatasql.connect(host=TD_HOST, user=TD_USER, password=TD_PASS) as con:
#     cur = con.cursor()

#     # 0) Quick sanity on source tables
#     item_cols  = get_columns(con, TD_DB, ITEM_EMB_SRC)
#     class_cols = get_columns(con, TD_DB, CLASS_EMB_SRC)
#     if "row_id" not in item_cols:
#         raise ValueError(f"{ITEM_EMB_SRC} must contain 'row_id'. Found: {item_cols[:10]}…")
#     if '"class"' not in [c.lower() if c==c.lower() else c for c in class_cols]:
#         # Some environments store quoted identifiers differently; just warn.
#         pass

#     item_feats  = sorted_vcols(item_cols)
#     class_feats = sorted_vcols(class_cols)
#     feats       = [c for c in item_feats if c in class_feats]
#     if not feats:
#         raise ValueError("No shared v* feature columns between items and classes.")

#     vec_cols        = ", ".join(feats)
#     vec_cols_quoted = ", ".join(f"'{c}'" for c in feats)

#     print(f"Shared features: {len(feats)} dims (e.g., {feats[:10]})")

#     # 1) (Re)create the reference embedding table: class_id + vectors
#     #    Join class embeddings (by "class") to unique_classes (class_id, class)
#     try:
#         cur.execute(f"DROP TABLE {REF_EMB};")
#     except Exception:
#         pass

#     create_ref_sql = f"""
#     CREATE MULTISET TABLE {REF_EMB} AS
#     (
#       SELECT
#         uc.class_id,
#         {vec_cols}
#       FROM {CLASS_EMB_SRC} AS ce
#       JOIN {CLASS_MAP} AS uc
#         ON ce."class" = uc.class
#     ) WITH DATA
#     PRIMARY INDEX (class_id);
#     """
#     cur.execute(create_ref_sql)
#     print(f"✅ Built {REF_EMB} (class_id + {len(feats)} dims)")

#     # 2) (Re)create the RESULT table via TD_VectorDistance (cosine), top‑1 per row_id
#     try:
#         cur.execute(f"DROP TABLE {RESULT};")
#     except Exception:
#         pass

#     create_res_sql = f"""
#     CREATE MULTISET TABLE {RESULT} AS
#     (
#       SELECT
#         o.Target_ID    AS row_id,
#         o.Reference_ID AS class_id,
#         1 - o.Distance AS score
#       FROM TD_SYSFNLIB.TD_VectorDistance
#       (
#         ON (SELECT row_id, {vec_cols} FROM {ITEM_EMB_SRC}) AS TargetTable
#         ON (SELECT class_id, {vec_cols} FROM {REF_EMB})     AS ReferenceTable DIMENSION
#         USING
#           TargetIDColumn       ('row_id')
#           RefIDColumn          ('class_id')
#           TargetFeatureColumns ({vec_cols_quoted})
#           RefFeatureColumns    ({vec_cols_quoted})
#           DistanceMeasure      ('cosine')
#       ) AS o
#       QUALIFY ROW_NUMBER() OVER (PARTITION BY o.Target_ID ORDER BY o.Distance) = 1
#     ) WITH DATA
#     PRIMARY INDEX (row_id);
#     """
#     cur.execute(create_res_sql)
#     print(f"✅ Rebuilt {RESULT} with top‑1 cosine similarity")

# print("All done.")

In [60]:
print("Item first 10 cols:", item_cols[:10])
print("Class first 10 cols:", class_cols[:10])
print("Detected item v-cols (first 10):", item_feats[:10], "… total", len(item_feats))
print("Detected class v-cols (first 10):", class_feats[:10], "… total", len(class_feats))
print("Shared feats (first 10):", feats[:10], "… total", len(feats))


Item first 10 cols: ['row_id', 'v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8']
Class first 10 cols: ['class', 'v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8']
Detected item v-cols (first 10): ['v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9'] … total 1024
Detected class v-cols (first 10): ['v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9'] … total 1024
Shared feats (first 10): ['v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9'] … total 1024


In [ ]:
## Verify unicode on teradata session

# td_context = get_context()
# raw_connection = td_context.engine.raw_connection()
# print("Verifying the character set of the established session...")
# with raw_connection.cursor() as cur:
#     cur.execute("HELP SESSION;")
    
#     session_info = cur.fetchone()
    
#     if session_info:
#         character_set = session_info[5]
#         print(f"✅ Session Character Set is: {character_set}")
#     else:
#         print("❌ Could not retrieve session information.")

# raw_connection.close()

In [ ]:
# copy_to_sql(
#     df=cleaned_tdf,              
#     table_name="cleaned_data",
#     if_exists="replace"
# )

In [ ]:
# tdf_2 =  tdf[['Item_Name', 'class']]
# tdf_2
######
# cat=tdf.groupby(['class']).count()
# cat
#######

In [ ]:
## cleaning
# tdf = tdf.assign(Item_Name = tdf.Item_Name.str.lower())
# tdf_stripped = tdf.assign(Item_Name = tdf.Item_Name.str.strip())

# tdf = tdf_stripped.assign(
#     Item_Name = tdf.Item_Name.otranslate("!@#$%^&*()-_=+[]{};:',.<>?/\\|`~", "")
# )

In [ ]:
# remove_context()